# ดึงข้อมูลโครงข่ายถนนจริงจาก OpenStreetMap และคำนวณระยะทางถึงถนน (Road Centerline)

**วัตถุประสงค์:** แทนที่ตัวแปรระยะทางถึงถนนมิตรภาพแบบประมาณ (interpolated waypoint route) ด้วยระยะทางถึงโครงข่ายถนนจริงจาก OpenStreetMap (OSM) ผ่านไลบรารี `osmnx`

**อินพุตที่ต้องเตรียม:** ไฟล์ CSV พิกัด 32 ไซต์ ต้องมีคอลัมน์อย่างน้อย:
- `site_id` — รหัสไซต์ (เช่น จุดตรวจวัดอากาศ-1, survey-0)
- `lat` — ละติจูด (decimal degrees)
- `lon` — ลองจิจูด (decimal degrees)

**เอาต์พุต:** ไฟล์ CSV ที่มีระยะทาง (กม.) จากแต่ละไซต์ไปยัง (1) ถนนที่ใกล้ที่สุดในโครงข่าย OSM ทั้งหมด และ (2) ถนนมิตรภาพ/ทางหลวงหมายเลข 2 โดยเฉพาะ (ถ้าตรวจพบในพื้นที่)


## ขั้นตอนที่ 1: ติดตั้งไลบรารีที่จำเป็น

In [ ]:
!pip install osmnx geopandas shapely pyproj --quiet
print("ติดตั้งเสร็จแล้ว")


In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from google.colab import files

print("osmnx version:", ox.__version__)


## ขั้นตอนที่ 2: อัปโหลดไฟล์พิกัด 32 ไซต์

กดปุ่มด้านล่างแล้วเลือกไฟล์ CSV ที่มีคอลัมน์ `site_id`, `lat`, `lon`


In [ ]:
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"อัปโหลดไฟล์: {csv_filename}")


In [ ]:
sites = pd.read_csv(csv_filename)

# ตรวจสอบและปรับชื่อคอลัมน์อัตโนมัติ เผื่อไฟล์ใช้ชื่อคอลัมน์ต่างออกไป
col_map = {}
for c in sites.columns:
    cl = c.strip().lower()
    if cl in ['site_id', 'site', 'id', 'name', 'จุดตรวจวัด']:
        col_map[c] = 'site_id'
    elif cl in ['lat', 'latitude', 'ละติจูด']:
        col_map[c] = 'lat'
    elif cl in ['lon', 'lng', 'long', 'longitude', 'ลองจิจูด']:
        col_map[c] = 'lon'
sites = sites.rename(columns=col_map)

assert {'site_id', 'lat', 'lon'}.issubset(sites.columns), \
    f"ไม่พบคอลัมน์ที่จำเป็นครบถ้วน พบคอลัมน์: {list(sites.columns)}"

sites = sites.dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f"จำนวนไซต์ที่มีพิกัดครบถ้วน: {len(sites)}")
sites.head(10)


## ขั้นตอนที่ 3: กำหนดขอบเขตพื้นที่และดึงโครงข่ายถนนจาก OSM

ระยะทางเดิมในต้นฉบับมีตั้งแต่ 0.4 ถึง 74.2 กม. จึงต้องขยายกรอบพื้นที่ (bounding box) ให้ครอบคลุมมากพอ (buffer ~0.3 องศา ≈ 33 กม. รอบขอบเขตของจุดทั้งหมด)


In [ ]:
buffer_deg = 0.3
north = sites['lat'].max() + buffer_deg
south = sites['lat'].min() - buffer_deg
east = sites['lon'].max() + buffer_deg
west = sites['lon'].min() - buffer_deg

print(f"ขอบเขตพื้นที่ดึงข้อมูล: N={north:.3f}, S={south:.3f}, E={east:.3f}, W={west:.3f}")

# ดึงโครงข่ายถนนสายหลัก (ลดขนาดข้อมูลด้วยการดึงเฉพาะถนนระดับ trunk/primary/secondary)
# หากต้องการถนนทุกระดับ (รวมซอย) ให้เปลี่ยน network_type='drive'
custom_filter = '["highway"~"motorway|trunk|primary|secondary"]'

G = ox.graph_from_bbox(
    bbox=(north, south, east, west),
    custom_filter=custom_filter,
    simplify=True,
    retain_all=True
)
print(f"จำนวนโหนด: {len(G.nodes)}, จำนวนเส้นทาง (edges): {len(G.edges)}")


## ขั้นตอนที่ 4: แปลงพิกัดเป็นระบบ UTM (EPSG:32647) เพื่อคำนวณระยะทางเป็นเมตรอย่างแม่นยำ

ใช้ระบบพิกัด UTM Zone 47N (EPSG:32647) ตามที่ใช้ในงานวิเคราะห์เชิงพื้นที่ส่วนอื่นของโครงการ


In [ ]:
G_proj = ox.project_graph(G, to_crs='EPSG:32647')

sites_gdf = gpd.GeoDataFrame(
    sites,
    geometry=[Point(xy) for xy in zip(sites['lon'], sites['lat'])],
    crs='EPSG:4326'
).to_crs('EPSG:32647')

X = sites_gdf.geometry.x.values
Y = sites_gdf.geometry.y.values
print("แปลงพิกัดเสร็จแล้ว")


## ขั้นตอนที่ 5: คำนวณระยะทางจากแต่ละไซต์ไปยังถนนที่ใกล้ที่สุด (โครงข่ายทั้งหมด)


In [ ]:
nearest_edges, dists_m = ox.distance.nearest_edges(G_proj, X, Y, return_dist=True)

sites['dist_to_nearest_road_km'] = np.array(dists_m) / 1000
sites[['site_id', 'lat', 'lon', 'dist_to_nearest_road_km']].sort_values('dist_to_nearest_road_km')


## ขั้นตอนที่ 6: คำนวณระยะทางเฉพาะถึงถนนมิตรภาพ / ทางหลวงหมายเลข 2

กรองเฉพาะเส้นทางที่มีชื่อหรือหมายเลขทางหลวงตรงกับ "Mittraphap"/"มิตรภาพ" หรือ ref เป็น "2" หรือ "AH1" (Asian Highway 1 ทับเส้นทางเดียวกันในบางช่วง) แล้วคำนวณระยะทางเฉพาะกับกลุ่มนี้


In [ ]:
def is_mittraphap(data):
    name = data.get('name', '')
    ref = data.get('ref', '')
    names = name if isinstance(name, list) else [name]
    refs = ref if isinstance(ref, list) else [ref]
    names = [str(n) for n in names]
    refs = [str(r) for r in refs]
    name_match = any('mittraphap' in n.lower() or 'มิตรภาพ' in n for n in names)
    ref_match = any(r.strip() in ['2', 'AH1', 'AH 1'] for r in refs)
    return name_match or ref_match

mtr_edges = [(u, v, k) for u, v, k, data in G_proj.edges(keys=True, data=True) if is_mittraphap(data)]
print(f"จำนวนเส้นทางที่ตรงกับถนนมิตรภาพ/ทางหลวงหมายเลข 2: {len(mtr_edges)}")

if len(mtr_edges) == 0:
    print("⚠️ ไม่พบถนนที่ตรงกับเงื่อนไข ลองขยาย custom_filter ในขั้นตอนที่ 3 ให้ครอบคลุมถนนทุกระดับ (network_type='drive') แล้วรันใหม่")
else:
    G_mtr = G_proj.edge_subgraph([(u, v, k) for u, v, k in mtr_edges]).copy()
    nearest_edges_mtr, dists_m_mtr = ox.distance.nearest_edges(G_mtr, X, Y, return_dist=True)
    sites['dist_to_mittraphap_road_km'] = np.array(dists_m_mtr) / 1000
    display(sites[['site_id', 'lat', 'lon', 'dist_to_nearest_road_km', 'dist_to_mittraphap_road_km']].sort_values('dist_to_mittraphap_road_km'))


## ขั้นตอนที่ 7: ตรวจสอบผลลัพธ์เทียบกับค่าประมาณเดิมในต้นฉบับ (ถ้ามี)

ถ้ามีคอลัมน์ระยะทางเดิม (เช่น `old_road_distance_km`) ในไฟล์ที่อัปโหลด จะเปรียบเทียบให้อัตโนมัติ


In [ ]:
old_col_candidates = [c for c in sites.columns if 'old' in c.lower() and 'road' in c.lower()]
if old_col_candidates:
    old_col = old_col_candidates[0]
    compare_col = 'dist_to_mittraphap_road_km' if 'dist_to_mittraphap_road_km' in sites.columns else 'dist_to_nearest_road_km'
    corr = sites[[old_col, compare_col]].corr().iloc[0, 1]
    print(f"สหสัมพันธ์ระหว่างค่าเดิม ({old_col}) กับค่าใหม่ ({compare_col}): r = {corr:.3f}")
    display(sites[['site_id', old_col, compare_col]])
else:
    print("ไม่พบคอลัมน์ระยะทางเดิมในไฟล์ที่อัปโหลด ข้ามการเปรียบเทียบ")


## ขั้นตอนที่ 8: บันทึกผลลัพธ์และดาวน์โหลด


In [ ]:
output_filename = 'road_distance_osm_verified.csv'
sites.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"บันทึกไฟล์: {output_filename}")
files.download(output_filename)


## หมายเหตุสำคัญ

- ระยะทางที่คำนวณได้คือระยะทางแบบ **straight-line จากจุดไซต์ถึงเส้นทาง (edge) ที่ใกล้ที่สุดในโครงข่าย** ไม่ใช่ระยะทางตามถนน (network distance / driving distance) — สอดคล้องกับวิธี haversine distance แบบเดิมที่ใช้ใน manuscript (Section 2.9) เพื่อให้เปรียบเทียบกันได้ตรงไปตรงมา
- หากผลลัพธ์ `dist_to_mittraphap_road_km` มีค่าผิดปกติ (เช่น 0 แถวหรือค่าที่สูงผิดปกติ) ให้ตรวจสอบว่า OSM มีการติดแท็กชื่อถนนสายนี้ในพื้นที่ครบถ้วนหรือไม่ อาจต้องเพิ่มเงื่อนไขการกรอง (เช่น ตรวจสอบ `ref` ที่เป็นภาษาไทยหรือรหัสอื่น) โดยดูตัวอย่างแท็กจริงจาก `G_proj.edges(data=True)` เพิ่มเติม
- แนะนำให้เปิดดูแผนที่ผลลัพธ์ (เช่น plot ด้วย `ox.plot_graph` ร่วมกับจุดไซต์) เพื่อตรวจสอบด้วยสายตาว่าเส้นทางที่จับคู่มานั้นสมเหตุสมผลก่อนนำไปใช้ในรายงานฉบับสมบูรณ์
